# Physical-quantity probes

Linear and two-layer MLP probes regress ground-truth physical quantities from
frozen embeddings. Held-out `R²` measures how accurately the representation
retains positions, angles, and gripper state.

## Setup

Probes are fitted on training embeddings and evaluated on a separate split.
Angles use sine/cosine targets; vector quantities report the mean `R²` across
components. The MLP uses an internal training split for early stopping.

Download the [analysis data](../../README.md#pretrained-models) first.
Set `PROBES = ["linear"]` to run only the linear probe.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "planning/train.py").is_file())
PLANNING = ROOT / "planning"
sys.path.insert(0, str(PLANNING))

import numpy as np
import pandas as pd
from IPython.display import display
from analysis.probes import TARGETS, load_split, physical_targets, fit_probe

EXPORT_ROOT = PLANNING / "outputs/embeddings"
METHODS = {"smwm": "SMWM", "sigreg": "SIGReg", "forward_only": "Fwd.-only"}
ENVS = {"tworoom": "TwoRoom", "reacher": "Reacher", "pusht": "Push-T", "ogbcube": "Cube"}
PROBES = ["linear", "mlp"]
PROBE_LABELS = {"linear": "Linear probe", "mlp": "MLP probe"}

## Probe fitting

The linear probe is ridge regression with `α=10⁻³`. The MLP has hidden widths
256 and 128. Scaling is fitted on training examples and reused at evaluation.
See [probes.py](probes.py) for the implementation.

In [ ]:
rows = []
for env, environment in ENVS.items():
    for method, label in METHODS.items():
        directory = EXPORT_ROOT / env / method
        if not all((directory / f"{split}_embeddings.pt").is_file() for split in ["train", "eval"]):
            print(f"No exported embeddings for {environment}/{label}.")
            continue
        z_train, train_arrays = load_split(directory, "train")
        z_eval, eval_arrays = load_split(directory, "eval")
        train_targets = physical_targets(env, train_arrays)
        eval_targets = physical_targets(env, eval_arrays)
        for quantity in train_targets:
            for probe in PROBES:
                print(f"{environment}/{label}: {quantity}, {probe}")
                score = fit_probe(z_train, train_targets[quantity], z_eval, eval_targets[quantity], probe)
                rows.append(dict(environment=environment, quantity=quantity,
                                 method=label, probe=probe, r2=score))
scores = pd.DataFrame(rows)

## Results

Held-out `R²`, with the best score per row and probe in bold. Ties at three
decimal places are bolded together. The CSV retains full precision; the
LaTeX table follows the paper's format.

In [ ]:
if not scores.empty:
    row_order = [(environment, target[0]) for env, environment in ENVS.items() for target in TARGETS[env]]
    column_order = [(probe, method) for probe in PROBES for method in METHODS.values()]
    table = scores.pivot(index=["environment", "quantity"], columns=["probe", "method"], values="r2")
    table = table.reindex(index=pd.MultiIndex.from_tuples(row_order, names=["Environment", "Quantity"]),
                          columns=pd.MultiIndex.from_tuples(column_order, names=["Probe", "Method"]))

    def highlight_best(row):
        styles = pd.Series("", index=row.index)
        for probe in PROBES:
            group = PROBE_LABELS[probe]
            values = row[group].round(3)
            for method, value in values.items():
                if pd.notna(value) and value == values.max():
                    styles[(group, method)] = "font-weight: bold"
        return styles

    display_table = table.rename(columns=PROBE_LABELS, level=0)
    display(display_table.style.format("{:.3f}", na_rep="—")
            .apply(highlight_best, axis=1)
            .set_table_attributes('style="border-collapse: collapse; font-family: Georgia, serif;"')
            .set_table_styles([
                {"selector": "th", "props": [("font-weight", "normal"), ("padding", "5px 10px")]},
                {"selector": "td", "props": [("padding", "5px 10px")]},
                {"selector": "thead", "props": [("border-top", "2px solid"), ("border-bottom", "1px solid")]},
                {"selector": "tbody", "props": [("border-bottom", "2px solid")]},
            ]))

    output = PLANNING / "outputs/analysis"
    output.mkdir(parents=True, exist_ok=True)
    scores.to_csv(output / "physical_probes.csv", index=False)

    method_count = len(METHODS)
    columns = "ll" + r"@{\hspace{10pt}}".join("r" * len(METHODS) for _ in PROBES)
    lines = [
        r"\begin{table}[t]", r"\centering",
        r"\caption{Held-out $R^2$ for physical quantities decoded from frozen embeddings.}",
        r"\label{tab:physical_quantity_probes}", r"\small",
        r"\setlength{\tabcolsep}{3.5pt}",
        r"\begin{tabular}{" + columns + "}", r"\toprule",
        "Environment & Quantity & " + " & ".join(
            r"\multicolumn{" + str(method_count) + "}{c}{" + PROBE_LABELS[probe] + "}" for probe in PROBES) + r" \\",
        "".join(r"\cmidrule(lr){" + f"{3 + method_count*i}-{2 + method_count*(i+1)}" + "}" for i in range(len(PROBES))),
        " & & " + " & ".join(method for _ in PROBES for method in METHODS.values()) + r" \\",
        r"\midrule",
    ]
    previous_environment = None
    for (environment, quantity), row in table.iterrows():
        first_in_group = environment != previous_environment
        if first_in_group and previous_environment is not None:
            lines.append(r"\midrule")
        values = []
        for probe in PROBES:
            best = row[probe].round(3).max()
            for method in METHODS.values():
                value = row[(probe, method)]
                text = f"{value:.3f}" if pd.notna(value) else "—"
                if pd.notna(value) and round(value, 3) == best:
                    text = r"\textbf{" + text + "}"
                values.append(text)
        label = environment if first_in_group else ""
        lines.append(f"{label} & {quantity} & " + " & ".join(values) + r" \\")
        previous_environment = environment
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    (output / "physical_probes.tex").write_text("\n".join(lines) + "\n")
else:
    print("Download or export embeddings before fitting probes.")